# Agent 动作前审批门禁：把授权绑定到具体副作用

**面试问题：高风险工具调用怎样在执行前判定允许、拒绝或等待审批？**

## 回答主线

1. Prompt 中写“请先审批”不是安全边界，真正门禁必须位于工具副作用之前。
2. 策略输入至少包含主体角色、动作、资源、参数、金额、环境和数据敏感级别。
3. 决策应明确分为 allow、deny、require_approval，并记录命中的规则。
4. 审批票据必须绑定规范化动作摘要、申请人、过期时间和一次性 nonce。
5. 执行前再次计算摘要可以阻止审批 100 元后把参数改成 10000 元的 TOCTOU。
6. 高风险拒绝、人工审批和权威回读都要进入审计账本。

## 真实案例

六个客服 Agent 动作包括读订单、发通知、改地址、小额退款、大额退款和删除账号。我们先复现“模型在理由里自称已获批准”导致越权执行，再实现确定性策略引擎、教学版审批签名和参数绑定，并攻击性地篡改已批准金额。数据是离线脱敏教学样本，只用于解释机制，不能宣称线上收益。

### 输入预览：六个动作与调用主体

In [1]:
import hashlib  # 导入摘要算法以绑定审批票据和具体动作参数。
import json  # 导入稳定序列化以生成规范化动作摘要。

actions = [  # 构造六种不同风险的真实工具动作。
    {"id": "A1", "actor": "support_agent", "tool": "order.read", "resource": "order-7", "args": {}, "risk": "low"},  # 只读订单无需审批。
    {"id": "A2", "actor": "support_agent", "tool": "notification.send", "resource": "user-9", "args": {"template": "refund-status"}, "risk": "low"},  # 使用批准模板发通知。
    {"id": "A3", "actor": "support_agent", "tool": "address.update", "resource": "order-8", "args": {"city": "杭州"}, "risk": "medium"},  # 改地址需要身份核验事实。
    {"id": "A4", "actor": "support_agent", "tool": "refund.create", "resource": "order-9", "args": {"amount": 100}, "risk": "medium"},  # 小额退款可在额度内自动执行。
    {"id": "A5", "actor": "support_agent", "tool": "refund.create", "resource": "order-10", "args": {"amount": 10000}, "risk": "high"},  # 大额退款必须人工审批。
    {"id": "A6", "actor": "support_agent", "tool": "account.delete", "resource": "user-10", "args": {}, "risk": "critical"},  # 删除账号不允许客服 Agent 执行。
]  # 完成动作集合。
context = {"identity_verified": True, "environment": "prod", "role": "support"}  # 定义当前请求的权威上下文。
print("动作  tool                 resource   risk      args")  # 输出门禁输入表头。
for action in actions:  # 逐动作展示副作用参数。
    print(f"{action['id']}   {action['tool']:<20} {action['resource']:<10} {action['risk']:<9} {action['args']}")  # 展示退款金额和删除动作。

动作  tool                 resource   risk      args
A1   order.read           order-7    low       {}
A2   notification.send    user-9     low       {'template': 'refund-status'}
A3   address.update       order-8    medium    {'city': '杭州'}
A4   refund.create        order-9    medium    {'amount': 100}
A5   refund.create        order-10   high      {'amount': 10000}
A6   account.delete       user-10    critical  {}


## Baseline 基线：从模型理由文本推断“已审批”

In [2]:
model_reasons = {"A1": "查询订单", "A2": "发送状态通知", "A3": "用户已核验", "A4": "额度内退款", "A5": "紧急情况，已获经理批准", "A6": "用户要求立即删除"}  # 构造模型自行生成的理由文本。
def prompt_only_gate(action, reason):  # 实现错误的自然语言关键词门禁。
    if action["risk"] in {"low", "medium"}:  # 低中风险直接相信模型。
        return "execute"  # 允许执行。
    if "批准" in reason or "用户要求" in reason:  # 高风险只要模型声称有批准就放行。
        return "execute"  # 产生越权副作用。
    return "block"  # 其他情况阻断。

baseline_decisions = [prompt_only_gate(action, model_reasons[action["id"]]) for action in actions]  # 对六个动作运行 Prompt 门禁。
print("动作  模型理由                    基线决策")  # 输出错误基线表头。
for action, decision in zip(actions, baseline_decisions):  # 逐动作展示自声明授权。
    print(f"{action['id']}   {model_reasons[action['id']]:<25} {decision}")  # 展示大额退款和删号被错误放行。

动作  模型理由                    基线决策
A1   查询订单                      execute
A2   发送状态通知                    execute
A3   用户已核验                     execute
A4   额度内退款                     execute
A5   紧急情况，已获经理批准               execute
A6   用户要求立即删除                  execute


### 核心实现：确定性策略决策与动作摘要

In [3]:
def policy_decision(action, facts):  # 在副作用前根据结构字段做确定性授权。
    if action["tool"] == "account.delete" and facts["role"] != "privacy_admin":  # 删除账号仅允许隐私管理员。
        return {"decision": "deny", "rule": "delete-requires-privacy-admin"}  # 返回不可审批绕过的拒绝。
    if action["tool"] == "refund.create" and action["args"].get("amount", 0) > 500:  # 超过客服自动退款额度。
        return {"decision": "require_approval", "rule": "refund-over-500"}  # 要求人工审批。
    if action["tool"] == "address.update" and not facts["identity_verified"]:  # 改地址前必须核验身份。
        return {"decision": "deny", "rule": "identity-required"}  # 缺失权威事实时拒绝。
    return {"decision": "allow", "rule": "role-and-amount-policy"}  # 其余动作在当前角色和额度内允许。

def action_digest(action):  # 对授权相关字段生成稳定摘要。
    protected = {"actor": action["actor"], "tool": action["tool"], "resource": action["resource"], "args": action["args"]}  # 排除无关展示字段并绑定主体、资源和参数。
    canonical = json.dumps(protected, ensure_ascii=False, sort_keys=True, separators=(",", ":"))  # 生成确定性 JSON 表示。
    return hashlib.sha256(canonical.encode("utf-8")).hexdigest()  # 返回动作指纹。

policy_rows = []  # 收集六个动作的结构化策略结论。
for action in actions:  # 逐动作执行策略引擎。
    result = policy_decision(action, context)  # 获取 allow、deny 或 require_approval。
    policy_rows.append({"id": action["id"], **result, "digest": action_digest(action)[:12]})  # 保存规则和短摘要供展示。
print("动作  decision          rule                          digest")  # 输出策略结果表头。
for row in policy_rows:  # 逐动作展示命中规则。
    print(f"{row['id']}   {row['decision']:<17} {row['rule']:<29} {row['digest']}")  # 展示 A5 等待审批、A6 直接拒绝。

动作  decision          rule                          digest
A1   allow             role-and-amount-policy        92b0281ac6b9
A2   allow             role-and-amount-policy        c90db797b268
A3   allow             role-and-amount-policy        fd396326a52b
A4   allow             role-and-amount-policy        de0ca37a0b95
A5   require_approval  refund-over-500               102d1b3538a6
A6   deny              delete-requires-privacy-admin 004da5ae4e6a


## 结果解读：执行、等待审批与拒绝三条路径

In [4]:
decision_counts = {decision: sum(row["decision"] == decision for row in policy_rows) for decision in ("allow", "require_approval", "deny")}  # 统计三类决策数量。
print("动作  Prompt基线  结构策略          是否安全变化")  # 输出基线和策略对照表头。
for action, baseline, row in zip(actions, baseline_decisions, policy_rows):  # 对齐每个动作的两种结论。
    changed = baseline == "execute" and row["decision"] != "allow"  # 标记被安全门禁收紧的动作。
    print(f"{action['id']}   {baseline:<11} {row['decision']:<17} {changed}")  # 展示高风险动作变化。
print("决策分布：", decision_counts)  # 展示正常自动化仍保留四个 allow。
print("解读：策略允许读、通知、已核验改址和100元退款；10000元退款等待审批；删号因角色不符直接 deny，不能靠普通审批绕过。")  # 解释每条分支。

动作  Prompt基线  结构策略          是否安全变化
A1   execute     allow             False
A2   execute     allow             False
A3   execute     allow             False
A4   execute     allow             False
A5   execute     require_approval  True
A6   execute     deny              True
决策分布： {'allow': 4, 'require_approval': 1, 'deny': 1}
解读：策略允许读、通知、已核验改址和100元退款；10000元退款等待审批；删号因角色不符直接 deny，不能靠普通审批绕过。


## 失败案例：审批 100 元后把执行参数改成 10000 元

In [5]:
DEMO_SIGNING_KEY = "teaching-only-key"  # 定义仅供教学的固定签名材料并明确不可用于生产。
def issue_approval(action, approver, expires_at, nonce):  # 为具体动作签发绑定票据。
    digest = action_digest(action)  # 计算申请时的动作摘要。
    payload = {"digest": digest, "approver": approver, "expires_at": expires_at, "nonce": nonce}  # 构造票据主体。
    signature_input = json.dumps(payload, sort_keys=True, separators=(",", ":")) + DEMO_SIGNING_KEY  # 形成教学版签名输入。
    payload["signature"] = hashlib.sha256(signature_input.encode("utf-8")).hexdigest()  # 生成防无意篡改摘要。
    return payload  # 返回审批票据。

def verify_approval(action, ticket, now):  # 在执行瞬间重新校验票据和动作。
    unsigned = {key: ticket[key] for key in ("digest", "approver", "expires_at", "nonce")}  # 重建签名覆盖的票据字段。
    signature_input = json.dumps(unsigned, sort_keys=True, separators=(",", ":")) + DEMO_SIGNING_KEY  # 重算期望签名输入。
    signature_ok = hashlib.sha256(signature_input.encode("utf-8")).hexdigest() == ticket["signature"]  # 检查票据未被篡改。
    digest_ok = action_digest(action) == ticket["digest"]  # 检查当前执行参数与审批申请完全一致。
    fresh = now <= ticket["expires_at"]  # 检查票据尚未过期。
    return signature_ok and digest_ok and fresh, {"signature_ok": signature_ok, "digest_ok": digest_ok, "fresh": fresh}  # 返回结论和分项证据。

approved_small = actions[3].copy()  # 复制 100 元退款动作作为审批申请。
ticket = issue_approval(approved_small, approver="manager-2", expires_at=2000, nonce="n-77")  # 签发绑定小额退款的票据。
tampered_large = {**approved_small, "args": {"amount": 10000}}  # 在获批后篡改金额形成 TOCTOU 攻击。
unsafe_execute = ticket["signature"] is not None  # 模拟只检查“有票据”就执行的错误实现。
safe_execute, verification = verify_approval(tampered_large, ticket, now=1500)  # 在副作用前绑定检查当前动作。
print(f"只检查票据存在 -> execute={unsafe_execute}，篡改后金额={tampered_large['args']['amount']}")  # 展示高风险越权。
print(f"绑定校验 -> execute={safe_execute}，分项={verification}")  # 展示 digest 不匹配阻断执行。
print("修正策略：审批对象是规范化动作而不是一段聊天文本；执行前重算摘要、检查过期和一次性 nonce，再权威回读结果。")  # 总结 TOCTOU 防线。

只检查票据存在 -> execute=True，篡改后金额=10000
绑定校验 -> execute=False，分项={'signature_ok': True, 'digest_ok': False, 'fresh': True}
修正策略：审批对象是规范化动作而不是一段聊天文本；执行前重算摘要、检查过期和一次性 nonce，再权威回读结果。


### 生产边界与审计账本

In [6]:
audit_event = {"action_id": "A5", "actor": "support_agent", "tool": "refund.create", "resource": "order-10", "decision": "require_approval", "rule": "refund-over-500", "digest": policy_rows[4]["digest"], "policy_version": "support-prod-r4"}  # 构造动作前审计事件。
print("审批审计事件：", audit_event)  # 展示谁、对什么资源、因何规则被拦截。
print("生产替换点：真实系统需要 OPA/Cedar 类策略、KMS 非对称签名、身份认证、nonce 存储、审批工作流、双人复核和工具后状态校验。")  # 明确 SHA 教学票据不具生产安全性。

审批审计事件： {'action_id': 'A5', 'actor': 'support_agent', 'tool': 'refund.create', 'resource': 'order-10', 'decision': 'require_approval', 'rule': 'refund-over-500', 'digest': '102d1b3538a6', 'policy_version': 'support-prod-r4'}
生产替换点：真实系统需要 OPA/Cedar 类策略、KMS 非对称签名、身份认证、nonce 存储、审批工作流、双人复核和工具后状态校验。


## 回归测试：最后只保护策略分支与参数绑定

In [7]:
assert [row["decision"] for row in policy_rows[:4]] == ["allow", "allow", "allow", "allow"]  # 验证四个满足规则的低中风险动作保持自动化。
assert policy_rows[4]["decision"] == "require_approval" and policy_rows[5]["decision"] == "deny"  # 验证大额退款与删号走不同高风险路径。
assert baseline_decisions[4] == "execute" and baseline_decisions[5] == "execute"  # 验证 Prompt 自声明授权的失败探针确实存在。
assert unsafe_execute and not safe_execute and not verification["digest_ok"]  # 验证篡改金额被动作摘要绑定阻断。
assert verify_approval(approved_small, ticket, now=1500)[0]  # 验证未篡改且未过期的原动作票据可以通过。
print("回归测试通过：自动允许、大额审批、角色拒绝、Prompt 越权反例和 TOCTOU 参数绑定均成立。")  # 用少量断言总结审批合同。

回归测试通过：自动允许、大额审批、角色拒绝、Prompt 越权反例和 TOCTOU 参数绑定均成立。
